# 01. Ingesta de datos
**Objetivo:** descargar los CSV desde https://mibici.net/es/datos-abiertos con un rango de fecha establecido, identificar diferentes esquemas y codificación entre los diferentes CSVs.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import requests

from mi_bici import data as ds
from mi_bici.config import RAW_DATA_DIR

pd.set_option('display.max_colwidth', None)


2026-10-16 22:40:53.623 | INFO     | mi_bici.config:<module>:11 - PROJ_ROOT path is: /home/chuy/Documents/mna/mlops/TC5061-Proyecto-Equipo11


## 1. Descargar datos

In [2]:
# Descargar un archivo de prueba

url = 'https://mibici.net/site/assets/files/61450/datos_abiertos_2024_01.csv'

try:
    with requests.get(url, stream=True, timeout=30) as response:
        response.raise_for_status()
except requests.HTTPError as error:
    print(f'{type(error).__name__}: {error}')

HTTPError: 406 Client Error: Not Acceptable for url: https://mibici.net/site/assets/files/61450/datos_abiertos_2024_01.csv


In [3]:
# ¿Porqué no se pueden descargar los archivos?

user_agents = ['python-requests/2.32', ds.HTTP_HEADERS['User-Agent'], 'Mozilla/5.0 (X11; Linux x86_64)']

rows = []
for user_agent in user_agents:
    with requests.get(url, headers={'User-Agent': user_agent}, stream=True, timeout=30) as response:
        rows.append({'user_agent': user_agent, 'status_code': response.status_code})

pd.DataFrame(rows)

,user_agent,status_code
0,python-requests/2.32,406
1,mi-bici-tc5061-equipo11,200
2,Mozilla/5.0 (X11; Linux x86_64),200


In [4]:
# Definir rango
START = '2024-01'
END = '2026-09'

sources = ds.load_sources()
trip_urls = ds.select_trip_urls(sources, START, END)

print(f'Meses a descargar: {len(trip_urls)}')

Meses a descargar: 33


In [5]:
# Descargar archivos
stations_path = ds.download_stations(sources)
trip_paths = ds.download_trips(trip_urls)

print(f'Catálogo de estaciones: {stations_path.name}')
print(f'Meses descargados: {len(trip_paths)}')

Catálogo de estaciones: nomenclatura_2026_09.csv
Meses descargados: 33


In [6]:
# Revisar peso de los archivos

files = pd.DataFrame(
    [
        {'year_month': year_month, 'file': path.name, 'MB': path.stat().st_size / 1e6}
        for year_month, path in trip_paths.items()
    ]
)
sizes = files['MB']

print(f'Archivos por mes - Total: {files["MB"].sum():,.1f} MB')
print(f'Archivos por mes - Promedio: {files["MB"].mean():,.1f} MB, mínimo {files["MB"].min():,.1f}, máximo {files["MB"].max():,.1f}')
print(f'Catálogo de estaciones: {stations_path.stat().st_size / 1e3:,.1f} KB')

Archivos por mes - Total: 1,070.1 MB
Archivos por mes - Promedio: 32.4 MB, mínimo 25.4, máximo 42.6
Catálogo de estaciones: 40.8 KB


## 2. Viajes

In [7]:
# Ver muestra

ds.read_raw_csv(trip_paths[START], nrows=5, dtype=str)

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
0,31789291,640715,M,1984,2024-01-01 00:05:25,2024-01-01 00:29:37,294,271
1,31789292,2353010,F,1995,2024-01-01 00:07:23,2024-01-01 00:16:37,182,254
2,31789293,1556365,M,1984,2024-01-01 00:07:31,2024-01-01 00:16:37,182,254
3,31789294,2626233,M,1994,2024-01-01 00:07:56,2024-01-01 00:31:51,35,154
4,31789295,2602006,M,1998,2024-01-01 00:08:09,2024-01-01 00:31:51,35,154


In [8]:
# Revisar encoding

header_summary = ds.compare_headers(trip_paths)
print(header_summary['encoding'].value_counts().to_string())

encoding
utf-8-sig    18
cp1252       15


In [9]:
# Identificar diferencias en los encabezados de los archivos

print('Número de columnas por archivo:')
print(header_summary['n_columns'].value_counts().to_string())

header_variants = header_summary.groupby('header_variant').agg(
    n_files=('year_month', 'count'),
    year_months=('year_month', list),
    columns=('columns', 'first'),
)
header_variants

Número de columnas por archivo:
n_columns
8    33


,n_files,year_months,columns
header_variant,,,
0,31,"[2024-01, 2024-03, 2024-04, 2024-06, 2024-07, 2024-08, 2024-09, 2024-10, 2024-11, 2024-12, 2025-01, 2025-02, 2025-03, 2025-04, 2025-05, 2025-06, 2025-07, 2025-08, 2025-09, 2025-10, 2025-11, 2025-12, 2026-01, 2026-02, 2026-03, 2026-04, 2026-05, 2026-06, 2026-07, 2026-08, 2026-09]","(Viaje_Id, Usuario_Id, Genero, Año_de_nacimiento, Inicio_del_viaje, Fin_del_viaje, Origen_Id, Destino_Id)"
1,2,"[2024-02, 2024-05]","(Viaje_Id, Usuario_Id, Genero, A}äe_nacimiento, Inicio_del_viaje, Fin_del_viaje, Origen_Id, Destino_Id)"


In [10]:
# Identificar formatos de fecha

# Esta tabla nos ayudará para comparar un solo valor por archivo, en el mejor de los casos
# Como son millones de registros, comparar fecha por fecha va a consumir tiempo y recursos
DIGITS_TO_NINE = str.maketrans('0123456789', '9' * 10)

date_shapes = []
for year_month, path in trip_paths.items():
    raw = pd.read_csv(path, encoding=ds.detect_encoding(path), usecols=[4, 5], header=0, dtype=str)
    values = pd.concat([raw.iloc[:, 0], raw.iloc[:, 1]]).dropna()
    shapes = values.str.translate(DIGITS_TO_NINE)
    for shape, n_values in shapes.value_counts().items():
        shape_values = values[shapes == shape]

        first_part_len = len(shape) - len(shape.lstrip('9'))
        date_shapes.append(
            {
                'year_month': year_month,
                'shape': shape,
                'n_values': n_values,
                'max_part_1': shape_values.str.slice(0, first_part_len).astype(int).max(),
                'example': shape_values.iloc[0],
            }
        )

date_shapes = pd.DataFrame(date_shapes)
date_shapes.groupby('shape').agg(
    n_files=('year_month', 'nunique'),
    n_values=('n_values', 'sum'),
    max_part_1=('max_part_1', 'max'),
    example=('example', 'first'),
    year_months=('year_month', lambda s: ', '.join(s.unique())),
)

,n_files,n_values,max_part_1,example,year_months
shape,,,,,
99/99/9999 99:99,2,1245609,31,01/09/2024 10:00,"2024-09, 2024-10"
99/99/9999 9:99,2,371915,31,01/09/2024 0:00,"2024-09, 2024-10"
9999-99-99 99:99:99,31,23462512,2026,2024-01-01 00:05:25,"2024-01, 2024-02, 2024-03, 2024-04, 2024-05, 2024-06, 2024-07, 2024-08, 2024-11, 2024-12, 2025-01, 2025-02, 2025-03, 2025-04, 2025-05, 2025-06, 2025-07, 2025-08, 2025-09, 2025-10, 2025-11, 2025-12, 2026-01, 2026-02, 2026-03, 2026-04, 2026-05, 2026-06, 2026-07, 2026-08, 2026-09"


## 3. Estaciones

In [11]:
# Muestreo

ds.read_raw_csv(stations_path, nrows=5)

,id,name,latitude,longitude,dpcapacity
0,2,(GDL-001) C. Epigmenio Glez./ Av. 16 de Sept.,20.666378,-103.348820,15
1,3,(GDL-002) C. Colonias / Av. Niños héroes,20.667228,-103.366000,15
2,4,(GDL-003) C. Vidrio / Av. Chapultepec,20.667690,-103.368252,19
3,5,(GDL-004) C. Ghilardi /C. Miraflores,20.691847,-103.362549,19
4,6,(GDL-005) C. San Diego /Calzada Independencia,20.681158,-103.339363,11


In [12]:
# Estructura

raw_stations = ds.read_raw_csv(stations_path)

print(f'Encoding: {ds.detect_encoding(stations_path)}')
print(f'Filas: {len(raw_stations)}, columnas: {raw_stations.shape[1]}')
print(f'Columnas: {raw_stations.columns.tolist()}')
raw_stations.dtypes

Encoding: utf-8-sig
Filas: 484, columnas: 5
Columnas: ['id', 'name', 'latitude', 'longitude', 'dpcapacity']


id              int64
name           object
latitude      float64
longitude     float64
dpcapacity      int64
dtype: object

In [13]:
# Describe

raw_ids = raw_stations['id']

print(f'id únicos: {raw_ids.nunique()} de {len(raw_stations)}, rango {raw_ids.min()} a {raw_ids.max()}')
print(f'Nulos por columna: {raw_stations.isna().sum().to_dict()}')

raw_stations.describe(include='all').T

id únicos: 484 de 484, rango 2 a 519
Nulos por columna: {'id': 0, 'name': 0, 'latitude': 0, 'longitude': 0, 'dpcapacity': 0}


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,484.0,NaN,NaN,NaN,254.758264,148.293765,2.0,129.75,252.0,379.25,519.0
name,484,484,(GDL-001) C. Epigmenio Glez./ Av. 16 de Sept.,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
latitude,484.0,NaN,NaN,NaN,20.67852,0.022032,20.63613,20.665077,20.67439,20.688609,20.739523
longitude,484.0,NaN,NaN,NaN,-103.361871,0.031626,-103.433215,-103.384164,-103.362443,-103.33932,-103.297112
dpcapacity,484.0,NaN,NaN,NaN,16.882231,4.126942,0.0,15.0,15.0,19.0,39.0


In [14]:
# ¿Porqué dpcapacity tiene valor 0?

raw_stations[raw_stations['dpcapacity'] == 0]

,id,name,latitude,longitude,dpcapacity
94,99,ex-(ZPN-009) C. Del Corral / C. Nicolás Bravo,20.685108,-103.333746,0
104,109,ex-(ZPN-019) C. Cabecera Mpal. Juan Manuel / C.,20.723221,-103.385583,0
111,116,ex-(ZPN-026) C. 16 Sep. / C. Fray A. de Segovia,20.686920,-103.334670,0
129,138,ex-(ZPN-044),20.697050,-103.363220,0


In [15]:
# ¿Qué es el prefijo ex-?

has_ex_prefix = raw_stations['name'].str.startswith('ex-')
zero_capacity = raw_stations['dpcapacity'] == 0

print(f'Estaciones con prefijo ex-: {has_ex_prefix.sum()}')
print(f'Con prefijo ex- y capacidad 0: {(has_ex_prefix & zero_capacity).sum()}')
print(f'Sin prefijo ex- y capacidad 0: {(~has_ex_prefix & zero_capacity).sum()}')

raw_stations[has_ex_prefix]

Estaciones con prefijo ex-: 14
Con prefijo ex- y capacidad 0: 4
Sin prefijo ex- y capacidad 0: 0


,id,name,latitude,longitude,dpcapacity
19,22,ex-(GDL-020) Juan Manuel / C.General Coronado,20.678714,-103.365730,15
92,97,ex-(ZPN-007) C. Libertad / Nicolás Bravo,20.726196,-103.392672,19
94,99,ex-(ZPN-009) C. Del Corral / C. Nicolás Bravo,20.685108,-103.333746,0
104,109,ex-(ZPN-019) C. Cabecera Mpal. Juan Manuel / C.,20.723221,-103.385583,0
108,113,ex-(ZPN-023) C. Juan Diego / C. Sarcófago,20.724890,-103.383614,11
109,114,ex-(ZPN-024) Manuel H. Alatorre / C. Industria,20.723560,-103.381901,11
111,116,ex-(ZPN-026) C. 16 Sep. / C. Fray A. de Segovia,20.686920,-103.334670,0
129,138,ex-(ZPN-044),20.697050,-103.363220,0
138,147,ex-(TLQ-006) C. Zalatitán / Av. Francisco Silva,20.641848,-103.302600,15
140,149,ex-(TLQ-008) C. Juárez / C.Francisco de Miranda,20.638410,-103.313340,15


In [16]:
# ¿Las siglas (GDL, ZPN, TLQ) son zonas?

zone_code = raw_stations['name'].str.extract(r'^(?:ex-)?\(?([A-Z]{3})-\d{3}\)?', expand=False)
print(f'Estaciones por zona: {zone_code.value_counts(dropna=False).to_dict()}')

# Nombres que no siguen el formato esperado.
standard_name = raw_stations['name'].str.match(r'^(?:ex-)?\([A-Z]{3}-\d{3}\) \S')
raw_stations.loc[~standard_name, ['id', 'name']]

Estaciones por zona: {'GDL': 357, 'ZPN': 110, 'TLQ': 17}


,id,name
18,21,(GDL-019) C. Juan Manuel / C. Andrés Terán
78,81,(GDL-079) C. Donato Guerra /Av. La Paz
129,138,ex-(ZPN-044)
153,162,(GDL-095) C. Mariano Bárcena / Maestros
228,237,(GDL-170) C. Relojeros/ C. Chamizal
240,250,ex-GDL-182 Av. Plan de San Luis / Av. Enríque
320,336,(GDL-243)Av. De los Maestros / C. Mil Cumbres
321,337,(GDL-245)C. Alejandrina / C. Torres Quintero
323,339,(ZPN-068)Francisco Mora / Aurelio Ortega
324,340,(ZPN-069)Jesús María Romo /Av. Aurelio Ortega


In [17]:
# ¿Porqué algunas zonas parece que tienen el formato correcto pero no cumplen con el patrón?

for name in raw_stations.loc[raw_stations['id'].isin([21, 81, 162, 237]), 'name']:
    print(repr(name))

'(GDL-019)  C. Juan Manuel / C. Andrés Terán'
'(GDL-079)  C. Donato Guerra /Av. La Paz'
'(GDL-095)  C. Mariano Bárcena / Maestros'
'(GDL-170)  C. Relojeros/ C. Chamizal'


## Hallazgos
- El servidor de MiBico bloquea los requests on el user agent default de `request`.

### Viajes (año_mes)
- El peso promedio de los archivos de cada año_mes es de **32.4 MB**.
- Se identifican 2 tipos de encodings: **utf-8-sig** (18 archivos) y **cp1252** (15 archivos).
- Se identifican 2 variantes de encabezados. La diferencia es en el nombre de la columna `Año_de_nacimiento`. Hay dos archivos con `A}äe_nacimiento`.
- El número de columnas (8) y el orden de las columnas se mantiene en los 33 archivos descargados.
- Se identifican 2 formatos diferente de fecha. **AAAA-MM-DD 00:00:00** (31 archivos) y **DD/MM/AAAA** (2 archivos).
- Los datos incluyen datos personales: `Usuario_Id`, `Genero` y `Año_de_nacimiento`.

### Estaciones
- 14 estaciones comienzan con el prefijo **ex-**. Suponemos que son estaciones retiradas.
- 4 estaciones tienen capacidad de 0. Estas estaciones también tienen el prefijo **ex-**.
- Todas las estaciones tienen un código de zona (GDL, ZPN, TLQ).
- Se identificaron diferentes formatos en los nombres de las estaciones:
  1. 1 espacio entre el código y dirección `(GDL-003) C. Vidrio / Av. Chapultepec`
  2. Sin paréntesis en el código `ex-GDL-182 Av. Plan de San Luis / Av. Enríque`
  3. Sin espacio entre el código y la dirección `(GDL-246)Juan José Baz/Silvestre Revueltas`
  4. Dos espacio entre el código y la dirección `(GDL-019)  C. Juan Manuel / C. Andrés Terán`
  5. Sin dirección, solo código `ex-(ZPN-044)`